# Araba Fiyatları (Car Prices)

🎯 Bu challenge’ın amacı, bir dataset hazırlamak ve şimdiye kadar öğrendiğiniz bazı feature selection tekniklerini uygulamaktır.

🚗 Arabalarla ilgili bir veri setiyle çalışıyoruz ve bir arabanın pahalı mı yoksa ucuz mu olduğunu tahmin etmek istiyoruz.

In [2]:
# Data manipulation
import numpy as np
import pandas as pd
# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns
# Sayısal bir özelliğin normal dağılım gösterip göstermediğini kontrol etme
from statsmodels.graphics.gofplots import qqplot


In [3]:
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Cars_dataset.csv"

❓ CSV dosyasını `df` adlı bir veri çerçevesine yükleyin.

In [4]:

df = pd.read_csv(url)
print("Veri setinin boyutu:", df.shape)
df.head()


Veri setinin boyutu: (205, 9)


,aspiration,enginelocation,carwidth,curbweight,enginetype,cylindernumber,stroke,peakrpm,price
0,std,front,64.1,2548,dohc,four,2.68,5000,expensive
1,std,front,64.1,2548,dohc,four,2.68,5000,expensive
2,std,front,65.5,2823,ohcv,six,3.47,5000,expensive
3,std,front,NaN,2337,ohc,four,3.40,5500,expensive
4,std,front,66.4,2824,ohc,five,3.40,5500,expensive


ℹ️ Dataset’in açıklaması [burada](https://drive.google.com/file/d/1ADSyjWfRGYqdXwCCN4PPC7PjQeMZ-ap-/view?usp=sharing ) mevcuttur. Egzersiz boyunca buna mutlaka referans verin.

## (1) Yinelenenler (Duplicates)

❓ Varsa, veri kümesinden yinelenenleri kaldırın. ❓

*Veri çerçevesini `df`* üzerine yazın.

In [5]:

# Yinelenen satır sayısını kontrol et
print("Yinelenen satır sayısı:", df.duplicated().sum())

# Yinelenen satırları kaldır
df = df.drop_duplicates().reset_index(drop=True)

# Temizlik sonrası kontrol
print("Temizlik sonrası yinelenen satır:", df.duplicated().sum())
print("Veri setinin yeni boyutu:", df.shape)


Yinelenen satır sayısı: 14
Temizlik sonrası yinelenen satır: 0
Veri setinin yeni boyutu: (191, 9)


## (2)  Eksik değerler (Missing values)

❓ Eksik değerleri bulun ve bunları ya `strategy = "most frequent"` (kategorik değişkenler için) ya da `strategy = "median"` (sayısal değişkenler için) kullanarak doldurun ❓

In [6]:

from sklearn.impute import SimpleImputer

# Eksik değerleri kontrol et
print("Eksik değerler:")
print(df.isnull().sum())

# Sayısal sütunları medyan ile doldur
numeric_cols = df.select_dtypes(include=["number"]).columns

numeric_imputer = SimpleImputer(strategy="median")
df[numeric_cols] = numeric_imputer.fit_transform(df[numeric_cols])

# Kategorik sütunları en sık değerle doldur
categorical_cols = df.select_dtypes(include=["object", "category"]).columns

categorical_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols])

print("\nTemizlik sonrası eksik değerler:")
print(df.isnull().sum())


Eksik değerler:
aspiration         0
enginelocation    10
carwidth           2
curbweight         0
enginetype         0
cylindernumber     0
stroke             0
peakrpm            0
price              0
dtype: int64

Temizlik sonrası eksik değerler:
aspiration        0
enginelocation    0
carwidth          0
curbweight        0
enginetype        0
cylindernumber    0
stroke            0
peakrpm           0
price             0
dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_15316\3365619002.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object", "category"]).columns


### `carwidth`

<details>
    <summary> 💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>carwidth</code> sütununda eksik değerler birden fazla şekilde temsil edilmektedir. Bazıları <code>np.nan</code>, bazıları ise <code>*</code> olarak yer alır. Bunlar tespit edildikten sonra, eksik değerler verinin %30’undan daha azını oluşturduğu için medyan değerle doldurulabilir.
</details>

In [8]:

# carwidth sütununu sayısal tipe dönüştür
df["carwidth"] = pd.to_numeric(df["carwidth"], errors="coerce")

# Eksik değerleri kontrol et
print("Önceki eksik değer:", df["carwidth"].isnull().sum())

# Eksik değerleri medyan ile doldur
df["carwidth"] = df["carwidth"].fillna(df["carwidth"].median())

# Sonucu kontrol et
print("Sonraki eksik değer:", df["carwidth"].isnull().sum())
print("Veri tipi:", df["carwidth"].dtype)


Önceki eksik değer: 4
Sonraki eksik değer: 0
Veri tipi: float64


### `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginelocation</code> kategorik bir feature olduğundan ve kategorilerin büyük çoğunluğu <code>front</code> olduğu için, en sık görülen değerle doldurun.
</details>

In [9]:

# Motor konumlarının dağılımını incele
print("Motor konumları:")
print(df["enginelocation"].value_counts(dropna=False))

# Eksik değerleri kontrol et
print("\nÖnceki eksik değer:", df["enginelocation"].isnull().sum())

# Eksik değerleri en sık görülen değerle doldur
mode_value = df["enginelocation"].mode(dropna=True)[0]
df["enginelocation"] = df["enginelocation"].fillna(mode_value)

# Sonucu kontrol et
print("\nSonraki eksik değer:", df["enginelocation"].isnull().sum())
print(df["enginelocation"].value_counts())


Motor konumları:
enginelocation
front    189
rear       2
Name: count, dtype: int64

Önceki eksik değer: 0

Sonraki eksik değer: 0
enginelocation
front    189
rear       2
Name: count, dtype: int64


🧪 **Kodunu test et**

In [10]:
from nbresult import ChallengeResult

result = ChallengeResult('missing_values',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts =============================
platform win32 -- Python 3.13.16, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\User\Desktop\S16D2-S-Data-car-prices\tests
plugins: platformdirs-4.14.0
collecting ... collected 2 items

test_missing_values.py::TestMissing_values::test_carwidth PASSED         [ 50%]
test_missing_values.py::TestMissing_values::test_engine_location PASSED  [100%]

============================== 2 passed in 0.71s ==============================


💯 You can commit your code:

git add tests/missing_values.pickle

git commit -m 'Completed missing_values step'

git push origin master



## (3) Sayısal özelliklerin ölçeklendirilmesi (Scaling the numerical features)

In [11]:
# Hatırlatma olarak, DataFrame hakkında bazı bilgiler
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   aspiration      191 non-null    str    
 1   enginelocation  191 non-null    str    
 2   carwidth        191 non-null    float64
 3   curbweight      191 non-null    float64
 4   enginetype      191 non-null    str    
 5   cylindernumber  191 non-null    str    
 6   stroke          191 non-null    float64
 7   peakrpm         191 non-null    float64
 8   price           191 non-null    str    
dtypes: float64(4), str(5)
memory usage: 13.6 KB


In [12]:
# Ve işte ölçeklendirmemiz gereken veri kümesinin sayısal özellikleri
numerical_features = df.select_dtypes(exclude=['object']).columns
numerical_features

C:\Users\User\AppData\Local\Temp\ipykernel_15316\3392077443.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are excluded by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `exclude` to remove them, or to `include` to keep them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  numerical_features = df.select_dtypes(exclude=['object']).columns


Index(['carwidth', 'curbweight', 'stroke', 'peakrpm'], dtype='str')

❓ **Soru: Sayısal feature’ların ölçeklenmesi** ❓

Sayısal feature’ları aykırı değerler (outliers) ve dağılımları açısından inceleyin ve duruma göre aşağıdaki yöntemleri uygulayın:
- Robust Scaler
- Standard Scaler

Dönüştürülmüş değerlerle orijinal sütunları değiştirin.

### `peakrpm` , `carwidth` , & `stroke`

<details>
    <summary>💡 <i>İpucu</i> </summary>

    
ℹ️ <code>peakrpm</code>, <code>carwidth</code> ve <code>stroke</code> normal dağılıma sahiptir ancak aynı zamanda bazı aykırı değerler (outlier) içerir. Bu nedenle `RobustScaler()` kullanılması tavsiye edilir.
</details>

In [13]:

from sklearn.preprocessing import RobustScaler, StandardScaler
import pandas as pd

# Ölçeklendirilecek sütunlar
cols = ["peakrpm", "carwidth", "stroke"]

# Sayısal tipe dönüştür
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Aykırı değerlere dayanıklı ölçeklendirme
scaler = RobustScaler()
df[cols] = scaler.fit_transform(df[cols])

# Sonuçları kontrol et
print(df[cols].describe())


          peakrpm    carwidth      stroke
count  191.000000  191.000000  191.000000
mean     0.018699    0.174714   -0.101309
std      0.674113    0.776462    1.059986
min     -1.357143   -1.925926   -4.066667
25%     -0.428571   -0.481481   -0.600000
50%      0.000000    0.000000    0.000000
75%      0.571429    0.518519    0.400000
max      2.142857    2.518519    2.933333


In [16]:

# Ölçeklendirme sonuçlarını kontrol et
df[["peakrpm", "carwidth", "stroke"]].head()


,peakrpm,carwidth,stroke
0,-0.142857,-0.518519,-2.033333
1,-0.142857,0.000000,0.600000
2,0.571429,0.370370,0.366667
3,0.571429,0.333333,0.366667
4,0.571429,0.296296,0.366667


### `curbweight`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>curbweight</code> normal bir dağılıma sahiptir ve aykırı değer (outlier) içermez. Bu nedenle Standard Scaler ile ölçeklenebilir.
</details>

In [14]:

# curbweight için StandardScaler
df["curbweight"] = pd.to_numeric(
    df["curbweight"], errors="coerce"
)

df["curbweight"] = df["curbweight"].fillna(
    df["curbweight"].median()
)

scaler = StandardScaler()
df[["curbweight"]] = scaler.fit_transform(
    df[["curbweight"]]
)

print(df["curbweight"].describe())


count    1.910000e+02
mean     3.720119e-17
std      1.002628e+00
min     -2.069633e+00
25%     -7.298694e-01
50%     -2.483172e-01
75%      7.462548e-01
max      2.846966e+00
Name: curbweight, dtype: float64


In [17]:

# Tüm ölçeklendirilmiş sütunları kontrol et
df[["peakrpm", "carwidth", "stroke", "curbweight"]].describe()


,peakrpm,carwidth,stroke,curbweight
count,191.000000,191.000000,191.000000,1.910000e+02
mean,0.018699,0.174714,-0.101309,3.720119e-17
std,0.674113,0.776462,1.059986,1.002628e+00
min,-1.357143,-1.925926,-4.066667,-2.069633e+00
25%,-0.428571,-0.481481,-0.600000,-7.298694e-01
50%,0.000000,0.000000,0.000000,-2.483172e-01
75%,0.571429,0.518519,0.400000,7.462548e-01
max,2.142857,2.518519,2.933333,2.846966e+00


🧪 **Kodunu test et**

In [18]:
from nbresult import ChallengeResult

result = ChallengeResult('scaling', dataset=df)
result.write()
print(result.check())


============================= test session starts =============================
platform win32 -- Python 3.13.16, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\User\Desktop\S16D2-S-Data-car-prices\tests
plugins: platformdirs-4.14.0
collecting ... collected 4 items

test_scaling.py::TestScaling::test_carwidth PASSED                       [ 25%]
test_scaling.py::TestScaling::test_curbweight PASSED                     [ 50%]
test_scaling.py::TestScaling::test_peakrpm PASSED                        [ 75%]
test_scaling.py::TestScaling::test_stroke PASSED                         [100%]

============================== 4 passed in 0.71s ==============================


💯 You can commit your code:

git add tests/scaling.pickle

git commit -m 'Completed scaling step'

git push origin master



## (4) Kategorik özelliklerin kodlanması (Encoding the categorical features)

❓ **Soru: Kategorik değişkenlerin encode edilmesi** ❓

👇 Encode edilmesi gereken feature’ları inceleyin ve duruma göre aşağıdaki teknikleri uygulayın:

- One-hot encoding
- Manuel ordinal encoding

DataFrame içinde, orijinal feature’ları encode edilmiş versiyonlarıyla değiştirin.

### `aspiration` & `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>aspiration</code> ve <code>enginelocation</code> ikili (binary) kategorik feature’lardır.
</details>

In [19]:

# Binary encoding
df["aspiration"] = df["aspiration"].map({
    "std": 0,
    "turbo": 1
})

df["enginelocation"] = df["enginelocation"].map({
    "front": 0,
    "rear": 1
})

# Sonuçları kontrol et
print(df[["aspiration", "enginelocation"]].head())


   aspiration  enginelocation
0           0               0
1           0               0
2           0               0
3           0               0
4           0               0


### `enginetype`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginetype</code> çok kategorili (multicategorical) bir feature’dır ve One-hot encoding uygulanmalıdır.
</details>

In [20]:

# enginetype için One-Hot Encoding
df = pd.get_dummies(
    df,
    columns=["enginetype"],
    dtype=int
)

# Oluşturulan yeni sütunları kontrol et
print(df.filter(like="enginetype").head())

# Veri setinin yeni boyutu
print("Veri setinin boyutu:", df.shape)


   enginetype_dohc  enginetype_dohcv  enginetype_l  enginetype_ohc  \
0                1                 0             0               0   
1                0                 0             0               0   
2                0                 0             0               1   
3                0                 0             0               1   
4                0                 0             0               1   

   enginetype_ohcf  enginetype_ohcv  enginetype_rotor  
0                0                0                 0  
1                0                1                 0  
2                0                0                 0  
3                0                0                 0  
4                0                0                 0  
Veri setinin boyutu: (191, 15)


In [21]:
df.shape

(191, 15)

### `cylindernumber`

<details>
    <summary>💡 İpucu </summary>

ℹ️ <code>cylindernumber</code> sıralı (ordinal) bir feature’dır ve sayısal değerlere manuel olarak encode edilmelidir.

</details>

In [22]:

# Silindir sayılarını sayısal değerlere dönüştür
cylinder_mapping = {
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "eight": 8,
    "twelve": 12
}

df["cylindernumber"] = df["cylindernumber"].replace(cylinder_mapping)

# Sayısal veri tipine dönüştür
df["cylindernumber"] = pd.to_numeric(
    df["cylindernumber"], errors="coerce"
)

# Sonuçları kontrol et
print(df["cylindernumber"].value_counts().sort_index())
print("Eksik değer:", df["cylindernumber"].isnull().sum())


cylindernumber
2       3
3       1
4     147
5      11
6      23
8       5
12      1
Name: count, dtype: int64
Eksik değer: 0


❓ Artık `cylindernumber`’ı 2 ile 12 arasında sayısal bir feature’a dönüştürdüğünüze göre, bunu ölçeklendirmeniz gerekiyor ❓

<br/>

<details>
    <summary>💡 İpucu </summary>

`cylindernumber`’ın mevcut dağılımına bakın ve kendinize şu soruları sorun:
- Ölçekleme, bir feature’ın dağılımını etkiler mi?
- Bu feature’ın dağılımına göre en uygun ölçekleme yöntemi hangisidir?
</details>

In [23]:

from sklearn.preprocessing import RobustScaler

# Silindir sayısını ölçeklendir
scaler = RobustScaler()

df[["cylindernumber"]] = scaler.fit_transform(
    df[["cylindernumber"]]
)

# Sonuçları kontrol et
print(df["cylindernumber"].describe())
print(df["cylindernumber"].value_counts().sort_index())


count    191.000000
mean       0.408377
std        1.095797
min       -2.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        8.000000
Name: cylindernumber, dtype: float64
cylindernumber
-2.0      3
-1.0      1
 0.0    147
 1.0     11
 2.0     23
 4.0      5
 8.0      1
Name: count, dtype: int64


<details>
<summary><i>Ölçekleme ve encoding işlemlerinden sonra DataFrame’inizin nasıl görünmesi gerektiğine dair bir ekran görüntüsü aşağıdadır</i></summary>
    
    
<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/02-Prepare-the-dataset/car_price_after_scaling_and_encoding.png">    

</details>

### `price`

👇 Hedef `price`ı kodlayın.

<details>
    <summary>💡 İpucu </summary>
    <br>
    ℹ️ <code>price</code> target değişkendir ve LabelEncoder ile encode edilmelidir.
</details>

In [24]:

from sklearn.preprocessing import LabelEncoder

# Price değişkenini sayısal etiketlere dönüştür
encoder = LabelEncoder()

df["price"] = encoder.fit_transform(df["price"])

# Sonuçları kontrol et
print(df["price"].head())
print("Benzersiz sınıf sayısı:", df["price"].nunique())
print("Veri tipi:", df["price"].dtype)


0    1
1    1
2    1
3    1
4    1
Name: price, dtype: int64
Benzersiz sınıf sayısı: 2
Veri tipi: int64


🧪 **Kodunu test et**

In [25]:
from nbresult import ChallengeResult

result = ChallengeResult('encoding',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts =============================
platform win32 -- Python 3.13.16, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\User\Desktop\S16D2-S-Data-car-prices\tests
plugins: platformdirs-4.14.0
collecting ... collected 4 items

test_encoding.py::TestEncoding::test_aspiration PASSED                   [ 25%]
test_encoding.py::TestEncoding::test_enginelocation PASSED               [ 50%]
test_encoding.py::TestEncoding::test_enginetype PASSED                   [ 75%]
test_encoding.py::TestEncoding::test_price PASSED                        [100%]

============================== 4 passed in 0.71s ==============================


💯 You can commit your code:

git add tests/encoding.pickle

git commit -m 'Completed encoding step'

git push origin master



## (5) Temel Modelleme (Base Modelling)

👏 Veri kümesi ön işleme tabi tutuldu ve artık modele uyarlanmaya hazır. 

❓ **Soru: Bir classification modelini ilk kez değerlendirme** ❓

Ön işlenmiş bu dataset üzerinde bir `LogisticRegression` modeli için cross-validation çalıştırın ve elde edilen skoru `base_model_score` adlı değişkende saklayın.

In [26]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer

# Bağımsız ve bağımlı değişkenleri ayır
X = df.drop(columns=["price"])
y = df["price"]

# Kalan kategorik değişkenleri sayısallaştır
X = pd.get_dummies(X, dtype=int)

# Eksik veya sonsuz değerleri temizle
X = X.replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)

# Logistic Regression modelini oluştur
model = LogisticRegression(max_iter=5000)

# 5 katlı Cross-Validation
scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

# Ortalama skoru kaydet
base_model_score = scores.mean()

print("Cross-Validation skorları:", scores)
print("Base Model Score:", base_model_score)


Cross-Validation skorları: [0.82051282 0.81578947 0.92105263 0.78947368 0.86842105]
Base Model Score: 0.8430499325236166


🧪 **Kodunu test et**

In [27]:
from nbresult import ChallengeResult

result = ChallengeResult('base_model',
                         score = base_model_score
)

result.write()
print(result.check())


============================= test session starts =============================
platform win32 -- Python 3.13.16, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\User\Desktop\S16D2-S-Data-car-prices\tests
plugins: platformdirs-4.14.0
collecting ... collected 1 item

test_base_model.py::TestBase_model::test_base_model_score PASSED         [100%]

============================== 1 passed in 0.16s ==============================


💯 You can commit your code:

git add tests/base_model.pickle

git commit -m 'Completed base_model step'

git push origin master



## (6) Özellik Seçimi  (Feature Selection (with _Permutation Importance_))

👩🏻‍🏫 Bir feature’ın target’ı tahmin etmede gerçekten önemli olup olmadığını tespit etmenin güçlü bir yolu şudur:

1. Bir model çalıştırın ve skorunu ölçün  
2. Bu feature’ı karıştırın (shuffle edin), modeli tekrar çalıştırın ve skoru tekrar ölçün  
    - Eğer performans **belirgin şekilde düşerse**, bu feature önemlidir ve **çıkarılmamalıdır**
    - Eğer performans **çok fazla düşmezse**, bu feature **elenebilir**

❓ **Sorular** ❓

1. Modele en az bilgi katkısı sağlayan feature’ları tespit etmek için feature permutation uygulayın.
2. Model performansının belirgin şekilde düşmeye başladığını fark edene kadar zayıf feature’ları dataset’ten çıkarın.
3. Elde ettiğiniz yeni güçlü feature set’i ile yeni bir modeli cross-validation ile değerlendirin ve skorunu `strong_model_score` adlı değişkende saklayın.

In [28]:

import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Özellikler ve hedef
X = pd.get_dummies(
    df.drop(columns=["price"]), dtype=int
)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)
y = df["price"]

# Model
def create_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=5000)
    )

# Permutation Importance için veri bölme
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3,
    random_state=42, stratify=y
)

model = create_model()
model.fit(X_train, y_train)

# Özelliklerin önemini hesapla
perm = permutation_importance(
    model, X_test, y_test,
    n_repeats=5,
    random_state=42,
    scoring="accuracy"
)

importance = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

print("Özellik önem sıralaması:")
print(importance)

# Güçlü özellikleri seç
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

feature_counts = sorted(set(
    [5, 10, 15, 20, len(X.columns)]
))
feature_counts = [
    n for n in feature_counts if n <= len(X.columns)
]

results = []

for n in feature_counts:
    selected = importance.head(n).index

    score = cross_val_score(
        create_model(),
        X[selected], y,
        cv=cv, scoring="accuracy"
    ).mean()

    results.append((n, score))

results_df = pd.DataFrame(
    results, columns=["Features", "Score"]
)

print("\nModel karşılaştırması:")
print(results_df)

# En yüksek skorlu özellik grubunu seç
best_n = int(
    results_df.loc[
        results_df["Score"].idxmax(), "Features"
    ]
)

strong_features = importance.head(best_n).index
X_strong = X[strong_features]

strong_model_score = cross_val_score(
    create_model(),
    X_strong, y,
    cv=cv, scoring="accuracy"
).mean()

print("\nSeçilen özellik sayısı:", best_n)
print("Strong Model Score:", strong_model_score)


Özellik önem sıralaması:
curbweight          0.162069
carwidth            0.075862
stroke              0.041379
enginetype_ohc      0.013793
enginetype_ohcf     0.013793
enginelocation      0.000000
enginetype_rotor    0.000000
enginetype_l        0.000000
enginetype_dohc     0.000000
enginetype_dohcv    0.000000
cylindernumber     -0.003448
enginetype_ohcv    -0.006897
aspiration         -0.010345
peakrpm            -0.017241
dtype: float64

Model karşılaştırması:
   Features     Score
0         5  0.911201
1        10  0.905938
2        14  0.900810

Seçilen özellik sayısı: 5
Strong Model Score: 0.9112010796221324


🧪 **Kodunu test et**

In [30]:

from nbresult import ChallengeResult

result = ChallengeResult(
    'strong_model',
    score=strong_model_score
)

result.write()
print(result.check())



============================= test session starts =============================
platform win32 -- Python 3.13.16, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\User\Desktop\S16D2-S-Data-car-prices\tests
plugins: platformdirs-4.14.0
collecting ... collected 1 item

test_strong_model.py::TestStrong_model::test_strong_model_score PASSED   [100%]

============================== 1 passed in 0.17s ==============================


💯 You can commit your code:

git add tests/strong_model.pickle

git commit -m 'Completed strong_model step'

git push origin master



## Bonus -  Verilerinizi sınıflandırma (Stratifying your data) ⚖️

💡 Veriyi training ve testing olarak bölerken, dataset’imizdeki kategorik değişkenlerin oranına dikkat etmemiz gerekir — ister target `y`’nin sınıfları olsun ister `X` içindeki kategorik bir feature olsun.

Aşağıda bir örneğe bakalım 👇

❓ Orijinal `X` ve `y` verinizi sklearn’in `train_test_split` fonksiyonunu kullanarak training ve testing olarak ayırın; karşılaştırılabilir sonuçlar elde etmek için `random_state=1` ve `test_size=0.3` kullanın.

In [31]:

from sklearn.model_selection import train_test_split

# Özellikleri ve hedefi ayır
X = df.drop(columns=["price"])
y = df["price"]

# Eğitim ve test verilerini oluştur
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1
)

print("Eğitim verisi:", X_train.shape)
print("Test verisi:", X_test.shape)


Eğitim verisi: (133, 14)
Test verisi: (58, 14)


❓ Training dataset’inizde ve testing dataset’inizde `price` sınıfı **1** olan araçların oranını kontrol edin.

> _Ham `df` içinde bu orana baktığınızda, yaklaşık **%50 / %50** olması gerekir._

In [32]:

# Orijinal veri setindeki price=1 oranı
original_ratio = (y == 1).mean() * 100

# Eğitim setindeki price=1 oranı
train_ratio = (y_train == 1).mean() * 100

# Test setindeki price=1 oranı
test_ratio = (y_test == 1).mean() * 100

print(f"Orijinal veri seti: %{original_ratio:.2f}")
print(f"Eğitim seti: %{train_ratio:.2f}")
print(f"Test seti: %{test_ratio:.2f}")


Orijinal veri seti: %50.79
Eğitim seti: %50.38
Test seti: %51.72


☝️ Hâlâ yaklaşık olarak **%50 / %50** civarında olmalı.

***Peki random state’i değiştirirsek ne olur?***

❓ `random_state` değerlerini **1’den 10’a** kadar döngüye alın ve her seferinde training ve testing dataset’lerindeki `price` sınıfı **1** olan araçların oranını hesaplayın. ❓

In [33]:

from sklearn.model_selection import train_test_split

for state in range(1, 11):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=state
    )

    train_ratio = (y_train == 1).mean() * 100
    test_ratio = (y_test == 1).mean() * 100

    print(
        f"Random State {state}: "
        f"Train = %{train_ratio:.2f}, "
        f"Test = %{test_ratio:.2f}"
    )


Random State 1: Train = %50.38, Test = %51.72
Random State 2: Train = %48.12, Test = %56.90
Random State 3: Train = %50.38, Test = %51.72
Random State 4: Train = %53.38, Test = %44.83
Random State 5: Train = %53.38, Test = %44.83
Random State 6: Train = %49.62, Test = %53.45
Random State 7: Train = %53.38, Test = %44.83
Random State 8: Train = %48.87, Test = %55.17
Random State 9: Train = %57.89, Test = %34.48
Random State 10: Train = %48.87, Test = %55.17


Her seferinde oranların değiştiğini, hatta bazen oldukça ciddi şekilde değiştiğini gözlemleyeceksiniz 😱! Bu durum model performansını etkileyebilir.

❓ `train_test_split(random_state=1)` kullanılarak eğitilen bir Logistic Regression modelinin test skorunu,  
`random_state=9` kullanılarak eğitilen modelin test skoru ile karşılaştırın ❓

Eğitimi training data üzerinde yapmayı ve skoru testing data üzerinde hesaplamayı unutmayın.

In [34]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Veri setini modellemeye hazırla
X_model = pd.get_dummies(X, dtype=int)
X_model = X_model.apply(pd.to_numeric, errors="coerce")
X_model = X_model.replace([np.inf, -np.inf], np.nan)

for state in [1, 9]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_model, y,
        test_size=0.3,
        random_state=state
    )

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=5000)
    )

    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)

    print(f"Random State {state} - Accuracy: {score:.4f}")


Random State 1 - Accuracy: 0.9483
Random State 9 - Accuracy: 0.8966


👀 `random_state=9` ile çok daha düşük bir skor görmelisiniz; çünkü bu test setindeki sınıf **1** araçların oranı %34.5 iken, training setinde bu oran %57.9’a, hatta orijinal dataset’te yaklaşık %50’ye yakındır.

Bu durum oldukça önemlidir; çünkü dataset’te oluşan bu **rastlantısal dengesizlik**, yalnızca model performansını düşürmekle kalmaz, aynı zamanda eğitim veya değerlendirme sırasında “gerçekliği” de bozabilir 🧐

***Peki bu sorunu nasıl çözebiliriz? Tren seti ve test seti arasında sınıfların dağılımını nasıl aynı tutabiliriz? 🔧***

🎁 Neyse ki sklearn’de, estimator (yani model) bir classifier olduğunda ve target bir sınıf olduğunda, bu durum `cross_validate` tarafından otomatik olarak ele alınır. 📚 [**sklearn.model_selection.cross_validate**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) dokümantasyonunda `cv` parametresini inceleyin.

Çözüm, aşağıdakini kullanmaktır:

> 📚 [**Stratification (Katmanlama)**](https://scikit-learn.org/stable/modules/cross_validation.html#stratification)

### Hedefin tabakalaşması (Stratification of the target)

💡 ***Stratification*** tekniğini `train_test_split` içinde de kullanabiliriz.

❓ Bu kez **1’den 10’a** kadar olan `random_state` döngüsünü tekrar çalıştırın, ancak bu sefer holdout yöntemine ***`stratify=y`*** parametresini de ekleyin. ❓

In [35]:

# Stratification ile sınıf dağılımını koru
for state in range(1, 11):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=state,
        stratify=y
    )

    train_ratio = (y_train == 1).mean() * 100
    test_ratio = (y_test == 1).mean() * 100

    print(
        f"Random State {state}: "
        f"Train = %{train_ratio:.2f}, "
        f"Test = %{test_ratio:.2f}"
    )


Random State 1: Train = %51.13, Test = %50.00
Random State 2: Train = %51.13, Test = %50.00
Random State 3: Train = %51.13, Test = %50.00
Random State 4: Train = %51.13, Test = %50.00
Random State 5: Train = %51.13, Test = %50.00
Random State 6: Train = %51.13, Test = %50.00
Random State 7: Train = %51.13, Test = %50.00
Random State 8: Train = %51.13, Test = %50.00
Random State 9: Train = %51.13, Test = %50.00
Random State 10: Train = %51.13, Test = %50.00


👀 Random state değişse bile, training ve testing verilerindeki sınıf oranları, orijinal `y` içindeki oranlarla aynı tutulur. İşte _stratification_ (katmanlama) tam olarak budur.

`train_test_split` fonksiyonunu `stratify` parametresiyle kullandığımızda, training ve testing verileri arasında **bir feature’ın oranlarını da koruyabiliriz**. Bu, özellikle aşağıdaki durumlarda son derece önemlidir:

- Churn tahmininde erkek ve kadın müşteri oranlarını korumak 🙋‍♂️ 🙋
- Ev fiyatlarını tahmin ederken büyük ve küçük evlerin oranlarını korumak 🏠 🏰
- Bir sonraki ürünü önerirken 1–5 arası review score dağılımını (multiclass!) korumak 🛍️
- vb.

Örneğin, bizim dataset’imizde `aspiration` feature’ının training ve testing verilerinde aynı oranda kalmasını istiyorsak, şu şekilde yazabiliriz:

`train_test_split(X, y, test_size=0.3, stratify=X.aspiration)`

---

Gördüğümüz gibi, **`cross_validate` [target değişkeni otomatik olarak stratify edebilir](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#:~:text=For%20int/None%20inputs%2C%20if%20the%20estimator%20is%20a%20classifier%20and%20y%20is%20either%20binary%20or%20multiclass%2C%20StratifiedKFold%20is%20used.)**, ancak **feature’lar için bunu yapmaz** 🤔 Bunun için biraz ekstra çalışmaya ihtiyacımız var.

Bunun için `StratifiedKFold` kullanmamız gerekiyor 🔬

### Tabakalaşma (Stratification - generalized)

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:

In [37]:

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np

skf = StratifiedKFold(n_splits=5)
scores = []

for train_idx, test_idx in skf.split(X_model, X["aspiration"]):
    X_train = X_model.iloc[train_idx]
    X_test = X_model.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=5000)
    )

    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))

print("Fold skorları:", scores)
print("Ortalama skor:", np.mean(scores))


Fold skorları: [0.9230769230769231, 0.7631578947368421, 0.9736842105263158, 0.7368421052631579, 0.9210526315789473]
Ortalama skor: 0.8635627530364374


📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:


🏁 Tebrikler! Tüm veri setini hazırladınız, özellik seçimi yaptınız ve hatta tabakalaşma hakkında bilgi edindiniz 💪.

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!